# Qwen3.5-9B · Real NextSearch Evidence-State QLoRA v0

This notebook uses **48 source-verified real search/fetch decisions**, balanced 24 continue / 24 final, from 28 original Apache-2.0 TRAIN tasks. All assistant targets and tool observations were originally recorded, not generated here.

**Research-only training**: no independent benchmark improvement has been shown. Original Qwen9B closed-loop NextSearch VAL was **0/8**; prior matched GAIA100 v3B was **18/100**.

The source JSONL is fixed to SHA256 `5420cfdb6111806a23b4bf6cb9cf04c77c75b5f0cf4af6c0b63727f312f40432`. Full 4096-token context is mandatory—never silently truncate. Use an L4/A100 GPU when available. A T4-compatible low-memory Q/V-only LoRA pilot is also included, but it may still run out of memory at 4,096 tokens. Failure stops without changing the source context; no T4 training success is claimed. No billing automatically occurs from this notebook; Colab runtime selection is yours.

[Source validation run](https://github.com/kolihhan/small-agent-qlora/actions/runs/43726123262) · [Source VAL baseline](https://github.com/kolihhan/small-agent-qlora/actions/runs/43722732105)

No old v3B or v7 checkpoint is overwritten.

In [ ]:
!pip -q install "transformers>=5.15,<6" "peft>=0.20,<1" "trl>=1.10,<2" "bitsandbytes>=0.50.1,<1" "accelerate>=1.10,<2" "datasets>=4" "jinja2>=3.1" "huggingface_hub>=1.0" "safetensors>=0.4.5"
!git clone -q --depth 1 -b experiment/nextsearch-gold-grounded-search-traces-20261115 https://github.com/kolihhan/small-agent-qlora.git /content/nextsearch-qlora-research
import sys
sys.path.insert(0, "/content/nextsearch-qlora-research/src")
print("Installed source-pinned QLoRA code; no training started.")

In [ ]:
import os, io, zipfile, hashlib, json, requests
from pathlib import Path
from google.colab import userdata
token = None
for secret_name in ("GITHUB_TOKEN", "GH_TOKEN"):
    try:
        token = userdata.get(secret_name)
    except Exception:
        token = None
    if token:
        break
if not token:
    raise RuntimeError("Set a GitHub token named GITHUB_TOKEN in Colab Secrets (read access to actions artifacts)")
artifact_id = 14166139977
url = f"https://api.github.com/repos/kolihhan/small-agent-qlora/actions/artifacts/{artifact_id}/zip"
response = requests.get(url, headers={"Authorization": f"Bearer {token}", "Accept":"application/vnd.github+json"}, timeout=120)
response.raise_for_status()
source_dir = Path("/content/nextsearch-v0-source")
source_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
    safe = set(zf.namelist())
    if "nextsearch-balanced-source-states-4k.jsonl" not in safe:
        raise RuntimeError("Frozen artifact content unexpected")
    zf.extract("nextsearch-balanced-source-states-4k.jsonl", source_dir)
source_file = source_dir/"nextsearch-balanced-source-states-4k.jsonl"
expected_sha = "5420cfdb6111806a23b4bf6cb9cf04c77c75b5f0cf4af6c0b63727f312f40432"
actual_sha = hashlib.sha256(source_file.read_bytes()).hexdigest()
if actual_sha != expected_sha:
    raise RuntimeError(f"Source SHA mismatch: {actual_sha}")
rows = [json.loads(s) for s in source_file.read_text().splitlines() if s.strip()]
from collections import Counter
states = Counter(r["capability"] for r in rows)
assert len(rows) == 48 and states == {"evidence_missing_continue":24, "evidence_present_finalize":24}
assert len({r["source_parent_sha256"] for r in rows}) == 28
print("Verified immutable 48 real source actions:", states)

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a Colab GPU runtime; this run cannot train on CPU"
device = torch.cuda.get_device_properties(0)
vram_gib = device.total_memory / 1024**3
print("GPU:", device.name, "| VRAM:", round(vram_gib, 1), "GiB")
# T4-class GPUs are an experimental low-memory route: restrict the learned
# LoRA matrices, NOT the original 4096-token evidence. Stop on OOM rather
# than silently shorten source history or substitute another model.
if vram_gib < 14.5:
    raise RuntimeError("Insufficient Colab GPU VRAM for a genuine 9B/4096 attempt")
LOW_VRAM = vram_gib < 21
LORA_TARGET_MODULES = "q_proj,v_proj" if LOW_VRAM else "all-linear"
PILOT_FLAVOR = "t4-qv" if LOW_VRAM else "l4-all"
print("Frozen trial:", PILOT_FLAVOR, "LoRA targets:", LORA_TARGET_MODULES)
print("No model training started; real OOM is possible on a T4.")

### Run eight small training steps

This is a **compatibility pilot**, not a promoted model. Save the adapter to a fresh Google Drive path to avoid losing it when Colab disconnects. The first time Colab asks to mount Drive, approve the standard Google Drive dialog. If the training is interrupted, inspect the last checkpoint before resuming; never claim eight completed steps from a partial checkpoint.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
from gaia_small_agent.training.qlora import train_qlora
from huggingface_hub import snapshot_download
PINNED_MODEL_REVISION = "c202236235762e1c871ad0ccb60c8ee5ba337b9a"
pinned_model_path = snapshot_download("Qwen/Qwen3.5-9B", revision=PINNED_MODEL_REVISION)
print("Using original Qwen model snapshot:", PINNED_MODEL_REVISION)
output = Path("/content/drive/MyDrive") / ("qlora-nextsearch-9b-" + PILOT_FLAVOR + "-v0")
if (output/"adapter_model.safetensors").exists():
    raise RuntimeError("Final adapter already exists; refusing to overwrite")
checkpoints = sorted(output.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
resume = str(checkpoints[-1]) if checkpoints else None
print("Resuming prior checkpoint:" if resume else "Starting new experiment:", resume or "fresh 0/8")
train_qlora(
    data_path=source_file,
    output_dir=output,
    model_name=pinned_model_path,
    max_length=4096,
    learning_rate=5e-5,
    lora_r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    max_steps=8,
    save_steps=4,
    gradient_accumulation_steps=1,
    lora_target_modules=LORA_TARGET_MODULES,
    exposure_count=None,
    resume_from_checkpoint=resume,
    require_full_prompt=True,
)
print("Training routine returned; verify files before interpreting result.")

In [ ]:
from pathlib import Path
import json, hashlib
needed = ["adapter_config.json","adapter_model.safetensors","run_config.json"]
missing = [name for name in needed if not (output/name).is_file()]
if missing:
    raise RuntimeError(f"No complete final 8-step adapter: missing {missing}. Look for checkpoint-4 or checkpoint-8 without claiming full completion.")
settings = json.loads((output/"run_config.json").read_text())
assert settings["max_steps"] == 8
assert settings["max_length"] == 4096
assert settings["training_examples"] == 48
assert settings["require_full_prompt"] is True
assert settings["lora_target_modules"] == (LORA_TARGET_MODULES if LORA_TARGET_MODULES == "all-linear" else LORA_TARGET_MODULES.split(","))
assert settings["resume_from_checkpoint"] == resume
print("CHECKPOINT VERIFIED",{"steps_requested":8,"data_sha256":expected_sha,"adapter_bytes":(output/"adapter_model.safetensors").stat().st_size,"gaia_score_changed":False,"promoted":False})

**After a verified adapter exists:** freeze the adapter SHA and run paired, tool-matched tests on the **same 8 independent NextSearch VALIDATION questions** and separate non-GAIA task families, then the protected GAIA100 split. Compare strict end-to-end accuracy, missing-evidence finals, tool calls, errors and resource use. Do not select checkpoints by peeking at VAL answers.